[![Open In Colab](./colab-badge.png)](https://colab.research.google.com/github/MooseNeuro/moose-notebooks/blob/main/Getting_started_with_MOOSE.ipynb) [![Binder](./binder_logo.png)](https://mybinder.org/v2/gh/MooseNeuro/moose-notebooks/HEAD?labpath=Getting_started_with_MOOSE.ipynb)

# Getting started


## Install pymoose with `pip`
The python module for `moose` is called `pymoose`. It is available on PyPI, so you can install it using `pip`

In [ ]:
## Only required on colab! Uncomment to enable.
# !pip install pymoose --quiet

## Import moose

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import moose


## Some builtin functions
- `moose.le(path)` : list elements under `path`
- `moose.showfield(path/object)` : show field names and values of object (at `path`)
- `moose.element(path)` : get the object at `path`
- `moose.connect(src, srcField, dest, destField)` : connect `srcField` on `src` object to `destField` on `dest` object


### `le()` function lists the elements under the current element
The output looks similar to directory listing on Unix systems. `/` is the root element, everything else is created under this. There are a bunch of elements that already exist. These elements are not for the user, but used by MOOSE for managing the infrastructure underlying the models and simulations.

In [ ]:
moose.le()

### Elements of class `Neutral` act as containers without any simulation related function
- Create an element `/data` for containing the data-collection tables
- Create another element `/model` for containing the model components
- These are just for convenience

In [ ]:
data = moose.Neutral('/data')
model = moose.Neutral('/model')

moose.le()

# Modelling a passive, single-compartment neuron
The `Compartment` class implements the basic passive neuronal compartment, which is an RC circuit with a battery representing the resting membrane potential in series with the membrane resistance.

![A passive neuronal compartment](neuronalcompartment.jpg)

In [ ]:
soma = moose.Compartment('/model/soma')

## Use `moose.showfield(...)` to see the fields and their default values

In [ ]:
moose.showfield(soma)

## Set the fields of the `Compartment`

In [ ]:
soma.initVm = -70e-3   # Membrane voltage at start of simulation
soma.Em = -70e-3       # Reversal potential
soma.Rm = 1e4          # Total membrane resistance
soma.Cm = 1e-7         # Total membrane capacitance

moose.showfield(soma)  # Check the updated field values

## Set up a current pulse generator `PulseGen`
The `PulseGen` class provides current pulses of specified amplitude and duration at regular interval

In [ ]:
inject = moose.PulseGen(f'{model.path}/inject')

`PulseGen` can be set up to generate arbitrary number of pulses. Two are included by default.

In [ ]:
inject.delay[0] = 10e-3      # first pulse should start at 10 ms
inject.width[0] = 20e-3      # width of first pulse
inject.level[0] = 1e-6       # amplitude of the current pulse

## Create `Table` objects for data recording
`Table` objects can accumulate a field value over time

In [ ]:
vm_tab = moose.Table('/data/Vm')
inject_tab = moose.Table('/data/inject')

## Connecting the model components with `moose.connect`
The different components of a model talk to each other via messages to exchange field values duing simulation. The connections for this exchange is set up via `moose.connect(src, srcField, dest, destField)`.

### Connect the `PulseGen` output as injection current into `Compartment`

In [ ]:
moose.connect(inject, 'output', soma, 'injectMsg')

### Connect the tables to the getters for fields to record

In [ ]:
moose.connect(vm_tab, 'requestOut', soma, 'getVm')
moose.connect(inject_tab, 'requestOut', inject, 'getOutputValue')

### *NOTE: Do not repeat `moose.connect(...)` between the same pair of fields on the same pair of objects* 
The `moose.connect(...)` calls should be executed exactly once. You may modify field values as many times as you wish and rerun the steps below to simulate the modified model. But repeating `moose.connect(source_element, source_message, target_element, target_message)` with the same parameters keeps increasing the number of connections, changing the stucture of the model, creating erroneous results.


### Initialize and run the simulation
Now we can tell moose to initialize all parameters to default or specified initial values (e.g., `initVm` of compartment will be assigned to `Vm`) using the `reinit()` function, and then start simulation for a specified runtime.

In [ ]:
runtime = 50e-3

moose.reinit()     # Assign the initial states

moose.start(runtime)

The `start(runtime)` function will run the simulation for `runtime` time, and at each timestep the table `vm_tab` will get the calculated value of `Vm`. The underlying array of `Vm` values can be accessed like a numpy array via the attribute `vector` of the `Table` object (`vm_tab`).

How about the time points? As mentioned earlier, successive timepoints of recording `Vm` are `vm_tab.dt` apart, i.e.,


$t_{0} = 0$, $t_{1} = dt$, $t_{2} = 2 dt$, $t_{3} = 3 dt$, ..., $t_{n-1} = (n - 1) dt$


So we can compute the time points (starting from 0) by multiplying the index of the data point by `dt`:
```
# numpy's arange(n) creates an array numbers containing 0 to (n-1)
t = np.arange(len(Vm)) * vm_tab.dt
```

In [ ]:
print('compartment dt',soma.dt)
print('table dt',vm_tab.dt)

In [ ]:
Vm = vm_tab.vector
current = inject_tab.vector
t = np.arange(len(Vm)) * vm_tab.dt

fig, axes = plt.subplots(nrows=2, sharex='all')
axes[0].plot(t * 1e3, Vm * 1e3, label='Voltage (mV)')
axes[1].plot(t * 1e3, current * 1e6, label='Injected current (uA)')
for ax in axes:
    ax.legend()

# Loading existing morphology and channels
A detailed neuron model may need hundreds of compartments and of the order of 10 ion channels. MOOSE provides utilities to load morphology traces. Moreover, it comes with a database of hundreds of Hodgkin-Huxley-type ion channels from [Ion Channel Genealogy](https://icg.neurotheory.ox.ac.uk/) and a small set of morphologies from published models.

## Deleting old models
We can use `moose.delete(path/object)` to remove an existing element along with its children.

In [ ]:
if moose.exists('/model'):
    moose.delete('/model')
    model = moose.Neutral('/model')
if moose.exists('/data'):
    moose.delete('/data')
    data = moose.Neutral('/data')

## Morphology database

In [ ]:
import moose.morphologies as morph

In [ ]:
morph.list()

In [ ]:
res = morph.load('purk_eds1994_full', f'{model.path}/Purkinje', RM=1.0, RA=1.0)

In [ ]:

for comp in moose.wildcardFind(f'{res.root.path}/##[TYPE=Compartment]'):
    comp.Em = -70e-3
    comp.initVm = -70e-3

## Display morphology
`pymoose` includes a couple of very basic functions to display morphology.

In [ ]:
from moose.plot_utils import plotMorphology, plotMorphologyGraph

In [ ]:
plotMorphologyGraph(res.root)

In [ ]:
plotMorphology(res.root)

## Ion channel prototypes from ICG
`moose.channels` submodule provides channel database and utility functions.

In [ ]:
import moose.channels as chan

### Searching channels by author, year, ion class
The database is large, one can search it with various parameters.

In [ ]:
chan.search(author='Traub', year='2003', ion_class='Na')

In [ ]:
chan.search(author='Traub', year='2003', ion_class='K')

### Inserting channels from ICG into passive multicompartmental model

In [ ]:
naf_list = chan.load(f'{res.root.path}/##[TYPE=Compartment]', 
                icg_id=1684, 
                gbar=lambda c: morph.surface_area(c) * 700.0, 
                Ek=50e-3)

In [ ]:
kdr_list = chan.load(f'{model.path}/##[TYPE=Compartment]', 
                     icg_id=1682, 
                     gbar=lambda c: morph.surface_area(c) * 400.0,
                     Ek=-100e-3)

In [ ]:
soma = moose.element(f'{res.root.path}/soma')
moose.le(soma)

## Setting up stimulus and data recording
### Create a `PulseGen` to inject current into soma

In [ ]:
inject2 = moose.PulseGen(f'{model.path}/inject2')

inject2.baseLevel = -0.1e-9
inject2.delay[0] = 50e-3
inject2.width[0] = 50e-3
inject2.level[0] = 0.2e-9

moose.connect(inject2, 'output', soma, 'injectMsg')

### Use HSolve for numerical integration

In [ ]:
solver = moose.HSolve(f'{res.root.path}/hsolve')
# solver.dt = 2.5e-6
solver.target = soma.path

### Setup voltage recording

In [ ]:
vm_soma = moose.Table(f'{data.path}/vm_soma')
moose.connect(vm_soma, 'requestOut', soma, 'getVm')

itab = moose.Table(f'{model.path}/Inject2')
moose.connect(itab, 'requestOut', inject2, 'getOutputValue')

## Initialize and run the simulation

In [ ]:
moose.reinit()

simtime = 150e-3
moose.start(simtime)

### Plot data

In [ ]:
t = np.linspace(0, simtime, len(itab.vector))
fig, axes = plt.subplots(nrows=2, sharex='all')
axes[0].plot(t * 1e3, vm_soma.vector * 1e3)
axes[1].plot(t * 1e3, itab.vector * 1e9)
axes[1].set_xlabel('Time (ms)')
axes[0].set_ylabel('Voltage (mV)')